In [6]:
import os
import re

import whisper
import yt_dlp


class ReadMateSTT:
    def __init__(self, model_size="base"):
        print(f">>> Whisper 모델({model_size}) 로딩 중...")
        self.model = whisper.load_model(model_size)
        self.temp_dir = "temp_audio"
        os.makedirs(self.temp_dir, exist_ok=True)

    def extract_from_youtube(self, url):
        """유튜브 URL에서 오디오만 추출하여 파일로 저장"""
        output_path = os.path.join(self.temp_dir, "target_audio")
        
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            'outtmpl': output_path,
            'quiet': True,
        }

        print(f">>> 유튜브 오디오 추출 시작: {url}")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        
        return f"{output_path}.mp3"

    def clean_text(self, text):
        """의미 없는 추임새 및 중복 제거"""
        fillers = r"(어\.\.\.|음\.\.\.|아\.\.\.|그\.\.\.|저\.\.\.|이제\s|사실\s|정말로\s)"
        cleaned = re.sub(fillers, "", text)
        cleaned = re.sub(r"\b(\w+)\s+\1\b", r"\1", cleaned)
        return re.sub(r"\s+", " ", cleaned).strip()

    def run_pipeline(self, youtube_url):
        """추출부터 정제까지 한 번에 실행"""
        try:
            # 1. 유튜브에서 mp3 추출
            audio_file = self.extract_from_youtube(youtube_url)
            
            # 2. STT 변환
            print(">>> 텍스트 변환 및 정제 시작...")
            result = self.model.transcribe(audio_file, fp16=False)
            
            # 3. 텍스트 정제
            raw_text = result['text']
            cleaned_text = self.clean_text(raw_text)
            
            print("\n" + "="*100)
            print("[최종 추출 결과]")
            print(cleaned_text[:300] + "...") # 앞부분 출력
            print("="*100)
            
            # 파일로 저장
            with open("lecture_ebs_minbyeongwon_output.txt", "w", encoding="utf-8") as f:
                f.write(cleaned_text)
                
            return cleaned_text

        except Exception as e:
            print(f"!!! 오류 발생: {e}")
        finally:
            # 사용 후 임시 오디오 파일 삭제
            if os.path.exists(audio_file):
                os.remove(audio_file)    

# --- 실제 실행부 ---
if __name__ == "__main__":
    # 1. 엔진 초기화
    stt_worker = ReadMateSTT()
    
    # 2. 테스트하고 싶은 유튜브 링크 입력
    test_url = "https://youtu.be/HIQd-UO4TQA?si=laKI41fe2auUu8cd"
    
    # 3. 파이프라인 가동
    stt_worker.run_pipeline(test_url)

>>> Whisper 모델(base) 로딩 중...
>>> 유튜브 오디오 추출 시작: https://youtu.be/HIQd-UO4TQA?si=laKI41fe2auUu8cd


>>> 텍스트 변환 및 정제 시작...                                      

[최종 추출 결과]
오세한이 아라는 말 자체에 어떤 의미가 들어가 있습니다. 바로 영어로 오션이라는 단어가 들어가 있습니다. 아주 넓은 큰 바다 대양이라는 뜻이 이미 들어가 있는 거죠. 그럼 결국 오세한이 아네 이 대양이 있는 그 지역에 땅이다 라는 것을 생각해 볼 수가 있겠습니다. 물론 그 지역 안에서는요. 오스트레일리아와 뉴질랜드가 주가 되는 건 맞습니다. 하지만 다른 여러 섬들까지 같이 포함되고 있습니다. 그러면 여러분들 하나 질문을 드릴게. 오세한이 아네는 대륙이다 아니다. 우리 뭐 5대형 6대 줄 알을 말 참 많이 하잖아요. 부가메리카, 나마...


In [ ]:
# 입력: 어떤 유튜브 링크든 run_pipeline(url)에 넣으면 바로 돌아가게 구현함
# 출력: 결과값으로 추임새가 다 제거된 정제된 텍스트가 나오고, stt_final_output.txt로 저장됨
# 환경: 라이브러리 yt-dlp, whisper, ffmpeg

In [ ]:
# 일반 예능 혹은 영상
# 인식 잘 못하는 이유 : 배경음악, 박수소리, 화자의 독특한 말투 등 때문 >> 소리 자체의 간섭이 정확도를 떨어뜨리는 주원인
# >> 범위 좁히기 제안 : '유튜브의 모든 영상을 다 변환하겠다' 보다는 '학습용 강의 만큼은 완벽하게 변환하겠다 는 목표가 현실적이고 기술적으로 구현 가능 !
# 학습용 강의는 보통 배경음악이 없고 강사의 발음 정확함 (여기에 우리가 특ㄷ정 과목 테마의 키워드를 힌트로 넣어주면 오타는 0의 수렴함 ..)

In [ ]:
from hanspell import (
    spell_checker,  # pip install git+https://github.com/ssut/py-hanspell.git
)


class ReadMateSTT:
    def __init__(self, model_size="base"):
        print(f">>> Whisper 모델({model_size}) 로딩 중...")
        self.model = whisper.load_model(model_size)
        self.temp_dir = "temp_audio"
        os.makedirs(self.temp_dir, exist_ok=True)

    def extract_from_youtube(self, url):
        output_path = os.path.join(self.temp_dir, "target_audio")
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            'outtmpl': output_path,
            'quiet': True,
        }
        print(f">>> 유튜브 오디오 추출 시작: {url}")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return f"{output_path}.mp3"

    def clean_text(self, text):
        """1단계: 규칙 기반 정제 + 2단계: AI 자동 맞춤법 교정"""
        # 1. 의미 없는 추임새 및 중복 제거
        fillers = r"(어\.\.\.|음\.\.\.|아\.\.\.|그\.\.\.|저\.\.\.|이제\s|사실\s|정말로\s)"
        text = re.sub(fillers, "", text)
        text = re.sub(r"\b(\w+)\s+\1\b", r"\1", text)
        text = re.sub(r"\s+", " ", text).strip()

        # 2. AI 자동 교정 (구경수탐 -> 국영수탐 등 문맥 교정)
        try:
            # hanspell은 글자 수 제한이 있으므로 안전하게 처리
            if len(text) > 0:
                spelled_sent = spell_checker.check(text)
                text = spelled_sent.checked
        except Exception as e:
            print(f">>> AI 교정 중 알림 (원본 유지): {e}")
        
        return text

    def run_pipeline(self, youtube_url):
        audio_file = None
        try:
            audio_file = self.extract_from_youtube(youtube_url)
            
            print(">>> 텍스트 변환 및 정제 시작...")
            # [방법 1] Initial Prompt 추가: 모델이 '국영수' 단어를 우선 인식하도록 유도
            # 수동으로 리스트를 짜지 않아도, 주요 키워드를 힌트로 주면 오타가 현저히 줄어듭니다.
            result = self.model.transcribe(
                audio_file, 
                fp16=False,
                initial_prompt="국영수탐, 수능특강, 모의고사, 사회탐구, 과학탐구, 교육과정"
            )
            
            raw_text = result['text']
            # [방법 2] 자동 교정 로직이 포함된 clean_text 호출
            cleaned_text = self.clean_text(raw_text)
            
            print("\n" + "="*100)
            print("[최종 추출 결과]")
            print(cleaned_text[:300] + "...")
            print("="*100)
            
            with open("lecture_output_automated.txt", "w", encoding="utf-8") as f:
                f.write(cleaned_text)
                
            return cleaned_text

        except Exception as e:
            print(f"!!! 오류 발생: {e}")
        finally:
            if audio_file and os.path.exists(audio_file):
                os.remove(audio_file)

if __name__ == "__main__":
    stt_worker = ReadMateSTT()
    test_url = "https://youtu.be/gGJXnf9phho?si=ZVO51onSRLLAnb3i"
    stt_worker.run_pipeline(test_url)

In [ ]:
# 불완전한 외부 라이브러리(hanspell) 제거
# 네이버 서버를 이용하는 py-hanspell에 의존 >> 이 라이브러리는 비공식 API라 서버가 응답하지 않거나, 파이썬 버전이 맞지 않으면 프로그램이 통째로 멈춰버리는(Freeze) 문제 발생

In [8]:

class ReadMateSTT:
    def __init__(self, model_size="base"):
        print(f">>> Whisper 모델({model_size}) 로딩 중...")
        self.model = whisper.load_model(model_size)
        self.temp_dir = "temp_audio"
        os.makedirs(self.temp_dir, exist_ok=True)

    def extract_from_youtube(self, url):
        output_path = os.path.join(self.temp_dir, "target_audio")
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            'outtmpl': output_path,
            'quiet': True,
        }
        print(f">>> 유튜브 오디오 추출 시작: {url}")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return f"{output_path}.mp3"

    def clean_text(self, text):
        """규칙 기반 정제 (불필요한 외부 라이브러리 제거)"""
        # 1. 의미 없는 추임새 제거
        fillers = r"(어\.\.\.|음\.\.\.|아\.\.\.|그\.\.\.|저\.\.\.|이제\s|사실\s|정말로\s)"
        text = re.sub(fillers, "", text)
        
        # 2. 반복되는 단어 제거
        text = re.sub(r"\b(\w+)\s+\1\b", r"\1", text)
        
        # 3. 공백 정렬
        return re.sub(r"\s+", " ", text).strip()

    def run_pipeline(self, youtube_url):
        audio_file = None
        try:
            audio_file = self.extract_from_youtube(youtube_url)
            
            print(">>> 텍스트 변환 및 정제 시작 (Initial Prompt 적용)...")
            
            # [핵심] 모델에게 직접 힌트를 주어 오타를 방지합니다. 
            # 여기에 '국영수탐'을 넣어두면 '구경수탐'으로 인식할 확률이 매우 낮아집니다.
            result = self.model.transcribe(
                audio_file, 
                fp16=False,
                initial_prompt="오세아니아"
            )
            
            raw_text = result['text']
            cleaned_text = self.clean_text(raw_text)
            
            print("\n" + "="*100)
            print("[최종 추출 결과]")
            # 텍스트가 비어있지 않은지 확인 후 출력
            if cleaned_text:
                print(cleaned_text[:500] + "...")
            else:
                print("추출된 텍스트가 없습니다. 오디오 상태를 확인해주세요.")
            print("="*100)
            
            with open("lecture_사회지리_민병권_3.txt", "w", encoding="utf-8") as f:
                f.write(cleaned_text)
                
            return cleaned_text

        except Exception as e:
            print(f"!!! 오류 발생: {e}")
        finally:
            if audio_file and os.path.exists(audio_file):
                os.remove(audio_file)

if __name__ == "__main__":
    stt_worker = ReadMateSTT()
    # 이지영 선생님 강의 링크로 테스트
    test_url = "https://youtu.be/HIQd-UO4TQA?si=laKI41fe2auUu8cd"
    stt_worker.run_pipeline(test_url)

>>> Whisper 모델(base) 로딩 중...
>>> 유튜브 오디오 추출 시작: https://youtu.be/HIQd-UO4TQA?si=laKI41fe2auUu8cd


>>> 텍스트 변환 및 정제 시작 (Initial Prompt 적용)...                

[최종 추출 결과]
오세아니아는 말 자체에 어떤 의미가 들어가 있습니까? 바로 영어로 오션이라는 단어가 들어가 있습니다. 아주 넓은 큰 바다 대양이라는 뜻이 이미 들어가 있는 거죠. 그럼 결국 오세아니아는 이 대양이 있는 그 지역에 땅이다 라는 것을 생각해 볼 수가 있겠습니다. 물론 그 지역 안에서는요, 오스테리아와 뉴질랜드가 주가되는 건 맞습니다. 하지만 다른 여러 섬들이까지 같이 포함되고 있습니다. 그러면 여러분들 하나 질문을 드릴게. 오세아니아는 대륙이다, 아니다. 우리 뭐 오대양 육대주를 말 참 많이 하잖아요. 뭐 부가메리카, 나마메리카, 유럽 아시아, 아프리카, 그 다음에 오세아니아 이렇게는 육대주라고 꼽습니다. 그런데 오세아니아는요, 엄밀하게 얘기하면 대륙은 아닙니다. 지금 바다와 어우러져 있는 섬들을 전체를 갖다가 오세아니아라고 하고 있고요. 대륙이라고 할 수 있는 것은 뭐가 있겠습니까? 바로 오스테리아가 있다라는 거죠. 여기 이 커다란 육지 부분은 대륙으로 보고요. 그 나머지는 전부 서음...


### 수정코드

In [9]:
import subprocess


class ReadMateSTT:
    def __init__(self, model_size="medium"):
        print(f">>> Whisper 모델({model_size}) 로딩 중...")
        self.model = whisper.load_model(model_size)
        self.temp_dir = "temp_audio"
        os.makedirs(self.temp_dir, exist_ok=True)

    def extract_from_youtube(self, url):
        """유튜브 URL에서 오디오만 추출하여 wav 파일로 저장"""
        output_path = os.path.join(self.temp_dir, "target_audio")
        
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'wav',
            }],
            'outtmpl': output_path,
            'quiet': True,
        }

        print(f">>> 유튜브 오디오 추출 시작: {url}")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        
        return f"{output_path}.wav"

    def preprocess_audio(self, input_path):
        """Whisper 최적 포맷으로 오디오 전처리 (16kHz, 모노, 음량 정규화)"""
        output_path = input_path.replace(".wav", "_processed.wav")
        print(">>> 오디오 전처리 중...")
        subprocess.run([
            "ffmpeg", "-i", input_path,
            "-ar", "16000",
            "-ac", "1",
            "-af", "loudnorm",
            output_path, "-y", "-q:a", "0"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return output_path

    def clean_text(self, text):
        """추임새 및 중복 제거"""
        # 단독 추임새만 제거 (의미 있는 단어 보호)
        cleaned = re.sub(r"\b(어+|음+|아+|그+|저+|에+)\b", "", text)
        return re.sub(r"\s+", " ", cleaned).strip()

    def run_pipeline(self, youtube_url):
        """추출부터 정제까지 한 번에 실행"""
        audio_file = None
        processed_file = None
        try:
            # 1. 유튜브에서 wav 추출
            audio_file = self.extract_from_youtube(youtube_url)
            
            # 2. 오디오 전처리
            processed_file = self.preprocess_audio(audio_file)

            # 3. STT 변환
            print(">>> 텍스트 변환 시작...")
            result = self.model.transcribe(
                processed_file,
                fp16=False,
                language="ko",
                beam_size=5,
                best_of=5,
                temperature=0.0,
                condition_on_previous_text=False,
                no_speech_threshold=0.6,
                compression_ratio_threshold=2.4,
            )
            
            # 4. 텍스트 정제
            raw_text = result['text']
            cleaned_text = self.clean_text(raw_text)
            
            print("\n" + "="*100)
            print("[최종 추출 결과]")
            print(cleaned_text[:300] + "...")
            print("="*100)
            
            # 파일로 저장
            with open("lecture_ebs_minbyeongwon_output.txt", "w", encoding="utf-8") as f:
                f.write(cleaned_text)
                
            return cleaned_text

        except Exception as e:
            print(f"!!! 오류 발생: {e}")
        finally:
            # 임시 파일 삭제
            for path in [audio_file, processed_file]:
                if path and os.path.exists(path):
                    os.remove(path)

# --- 실제 실행부 ---
if __name__ == "__main__":
    # VRAM 여유 있으면 large-v3 권장
    stt_worker = ReadMateSTT(model_size="medium")
    
    test_url = "https://youtu.be/HIQd-UO4TQA?si=laKI41fe2auUu8cd"
    
    stt_worker.run_pipeline(test_url)

>>> Whisper 모델(medium) 로딩 중...


100%|█████████████████████████████████████| 1.42G/1.42G [00:52<00:00, 29.3MiB/s]


>>> 유튜브 오디오 추출 시작: https://youtu.be/HIQd-UO4TQA?si=laKI41fe2auUu8cd


>>> 오디오 전처리 중...                                         
>>> 텍스트 변환 시작...

[최종 추출 결과]
자, 오세아니아라는 말 자체에 어떤 의미가 들어가 있습니까? 바로 영어로 오션이라는 단어가 들어가 있습니다. 아주 넓은 큰 바다, 대양이라는 뜻이 이미 들어가 있는 거죠. 그럼 결국 오세아니아는 이 대양이 있는 지역의 땅이다라는 것을 생각해 볼 수가 있겠습니다. 자, 물론 지역 안에서는요. 오스트레일리아와 뉴질랜드가 주가 되는 건 맞습니다. 하지만 다른 여러 섬들까지 같이 포함되고 있습니다. 자, 그러면 여러분들 하나 질문을 드릴게요. 오세아니아는 대륙이다 아니다? 자, 우리 뭐 5 대 6 대주라는 말 참 많이 하잖아요. 뭐 북아메...


In [10]:

class ReadMateSTT:
    def __init__(self, model_size="medium"):
        print(f">>> Whisper 모델({model_size}) 로딩 중...")
        self.model = whisper.load_model(model_size)
        self.temp_dir = "temp_audio"
        os.makedirs(self.temp_dir, exist_ok=True)

    def extract_from_youtube(self, url):
        """유튜브 URL에서 오디오만 추출하여 wav 파일로 저장"""
        output_path = os.path.join(self.temp_dir, "target_audio")
        
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'wav',
            }],
            'outtmpl': output_path,
            'quiet': True,
        }

        print(f">>> 유튜브 오디오 추출 시작: {url}")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        
        return f"{output_path}.wav"

    def preprocess_audio(self, input_path):
        """Whisper 최적 포맷으로 오디오 전처리 (16kHz, 모노, 음량 정규화)"""
        output_path = input_path.replace(".wav", "_processed.wav")
        print(">>> 오디오 전처리 중...")
        subprocess.run([
            "ffmpeg", "-i", input_path,
            "-ar", "16000",
            "-ac", "1",
            "-af", "loudnorm",
            output_path, "-y", "-q:a", "0"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return output_path

    def clean_text(self, text):
        """추임새 및 중복 제거"""
        # 단독 추임새만 제거 (의미 있는 단어 보호)
        cleaned = re.sub(r"\b(어+|음+|아+|그+|저+|에+)\b", "", text)
        return re.sub(r"\s+", " ", cleaned).strip()

    def run_pipeline(self, youtube_url):
        """추출부터 정제까지 한 번에 실행"""
        audio_file = None
        processed_file = None
        try:
            # 1. 유튜브에서 wav 추출
            audio_file = self.extract_from_youtube(youtube_url)
            
            # 2. 오디오 전처리
            processed_file = self.preprocess_audio(audio_file)

            # 3. STT 변환
            print(">>> 텍스트 변환 시작...")
            result = self.model.transcribe(
                processed_file,
                fp16=False,
                language="ko",
                beam_size=5,
                best_of=5,
                temperature=0.0,
                condition_on_previous_text=False,
                no_speech_threshold=0.6,
                compression_ratio_threshold=2.4,
            )
            
            # 4. 텍스트 정제
            raw_text = result['text']
            cleaned_text = self.clean_text(raw_text)
            
            print("\n" + "="*100)
            print("[최종 추출 결과]")
            print(cleaned_text[:300] + "...")
            print("="*100)
            
            # 파일로 저장
            with open("lecture_ebs_leejiyoung_output.txt", "w", encoding="utf-8") as f:
                f.write(cleaned_text)
                
            return cleaned_text

        except Exception as e:
            print(f"!!! 오류 발생: {e}")
        finally:
            # 임시 파일 삭제
            for path in [audio_file, processed_file]:
                if path and os.path.exists(path):
                    os.remove(path)

# --- 실제 실행부 ---
if __name__ == "__main__":
    # VRAM 여유 있으면 large-v3 권장
    stt_worker = ReadMateSTT(model_size="medium")
    
    test_url = "https://youtu.be/gGJXnf9phho?si=eptthIcRwI4jXbqM"
    
    stt_worker.run_pipeline(test_url)

>>> Whisper 모델(medium) 로딩 중...
>>> 유튜브 오디오 추출 시작: https://youtu.be/gGJXnf9phho?si=eptthIcRwI4jXbqM


>>> 오디오 전처리 중...                                           
>>> 텍스트 변환 시작...

[최종 추출 결과]
너가 수학과외쌤을 만나는 이유는 뭐야? 이러면 수학 잘하게 생겨가지고 그럼 너네 지금 나 이용해 먹는 거네 왜 웃지? 이상하네 너가 국영수 탐 중에 제일 자신 없는 과목이 뭐야? 그럼 너가 수학을 대형강의도 듣고 인강도 듣는데 너무너무 수학을 못 알아듣겠어서 이제 과외를 병행하려고 그래 수학과외쌤을 한 번 상상해봐요. 과외 받아본 적 있니? . 수학과외쌤 너희도 생각해봐. 너가 수학과외쌤을 만나는 이유는 뭐야? 좋아서 사랑했었어. 잠깐. 이름이 우진? 우진이는 천우진이 나쁜 애네. 왜 나쁜 애냐면 사람을 수단으로 이용했잖아. 수학성...
